In [ ]:
# Parameters
run_date = "2026-01-01"  # papermill replacement
import os
output_dir = os.environ.get("ORION_SIGNALS_DIR", "../signals")
dry_run = False

# ensure output exists
os.makedirs(output_dir, exist_ok=True)


In [ ]:
# Import basic modules
import pandas as pd
import numpy as np
import math
import datetime
from datetime import timedelta
from typing import Optional, List, Dict, Any

# Import warnings
import warnings
warnings.filterwarnings("ignore")


In [ ]:
from __future__ import annotations

# ── REVERSAL: fade the stack that is already extreme at 15:50 ──────────────────────────────────────────────
#
# THE TRADE (user, 2026-09-21): at 15:50 read the stack's own Stack% value. If |it| > 0.5, the stack is
# "high" (positive) or "low" (negative). Enter at 16:00 AGAINST that extreme - short the high one, long the
# low one - and hold to a class-specific exit: 18:00, 21:00, 04:00 (next day), 07:00 (next day) or 09:30
# (next day, named PRINT). Every one of those five is scored as its own class, per ticker, per sign of the
# 15:50 reading. PnL is the raw Stack% move from entry to exit (no beta/bench adjustment - "Рахуй піенел по
# значенню STAK%").
#
# NEAREST-MINUTE SNAPPING (user): a checkpoint with no print at its own minute reads the nearest one instead
# - explicitly +/-5 min for the 15:50 signal, "so do the same for the classes too". Every checkpoint below
# gets a tolerance this way, but never wide enough to reach into a NEIGHBOURING checkpoint's own territory -
# see `_bounded_tolerances`.
#
# MIDNIGHT (user: "реалізуй перехід через опівніч на наступний день"): 18:00/21:00 are the SAME calendar
# evening as the 16:00 entry; 04:00/07:00/09:30 are the FOLLOWING calendar morning. The stage module gives
# every one of them the same session id by splitting the day at 15:00 (well before the 15:50 signal), so a
# session runs from 15:00 one afternoon to 15:00 the next - see rv_cell_stage.py.
#
# `dev_sig`/`sigma` (the ArbitRage/PairFlux normalisation) do not apply here: the user's own instruction is
# to read the raw Stack% column and gate on a flat 0.5 pp trigger, so this file only ever sees Stack%.
import math
from collections import namedtuple

import numpy as np
import pandas as pd

SESSION_SPLIT_MIN = 900   # 15:00 - strictly before the 15:50 signal, so every checkpoint 15:50..next 09:30
                          # lands in one session (see rv_cell_stage.to_smin)

Checkpoint = namedtuple("Checkpoint", "name smin tol")

# (name, hh:mm on the checkpoint's own clock, the tolerance the user asked for before it is bounded)
#
# ENTRY IS 16:01, NOT 16:00 (user, 2026-09-22, checked against real tape): the user's very first instinct
# was "Заходимо в 16:01", then self-doubted it to 16:00 "щоб не було плутанини" - the real data settles it
# the other way. Queried the live bridge's own tape (/api/tape/query) minute-by-minute across three real
# dates (2026-09-10/15/18): LstPrcLstClsPct (Stack%) reads EXACTLY 0.0 at minute 960 (16:00) for the large
# majority of tickers checked (22/25) regardless of the ticker's real after-hours move - lstCls (the
# reference close Stack% is measured against) has not yet rolled over to TODAY's new close at that literal
# minute; it reliably has by 16:01. A checkpoint still sitting at 16:00 would have its nearest-search try
# that exact minute FIRST and silently accept the placeholder zero as "the entry price" for essentially
# every ticker, corrupting every trade's PNL and every downstream gamma/rating computed from it.
#
# This was the LIVE BRIDGE's own tape behaviour, verified against ITS OWN data. It does not, by itself,
# fix final.parquet's own artifact - see REBASE_TO_TODAYS_CLOSE below for that. Kept anyway: harmless even
# if final.parquet has no literal zero at :00, and keeps this notebook's clock identical to
# ReversalTiming.Default on the bridge, which DOES need it.
_RAW_CHECKPOINTS = (
    ("SIGNAL", (15, 50), 5),     # the 15:50 reading itself - the user's explicit +/-5
    ("ENTRY", (16, 1), 5),       # the price the position is taken at - not specified separately, so mirrored
    ("EXIT18", (18, 0), 10),
    ("EXIT21", (21, 0), 10),
    ("EXIT04", (4, 0), 10),
    ("EXIT07", (7, 0), 10),
    ("PRINT", (9, 30), 10),      # 09:30, renamed PRINT per the user's instruction
)
EXIT_CLASSES = ("EXIT18", "EXIT21", "EXIT04", "EXIT07", "PRINT")

# A trade's date label is the EXIT's own calendar date (user, 2026-09-22: "Маркування дати трейду завжди
# рахується по виходу... Якщо вхід був об 18:00 - 11 числа це трейд 11 числа" - EXIT18/EXIT21 close the SAME
# evening as the 16:00 entry, so they date-label like the entry (offset 0 from the session's own EVENING
# date); EXIT04/EXIT07/PRINT close the FOLLOWING morning, so they date-label one day later (offset 0 from the
# session's own MORNING date - which is what `sdate` already is, see rv_cell_stage.py's `late` split). Fixed
# per class, not derived from exit_minute's sign at runtime, so the mapping is visible in one place.
EXIT_DATE_OFFSET = {"EXIT18": -1, "EXIT21": -1, "EXIT04": 0, "EXIT07": 0, "PRINT": 0}


def _to_smin(hm, split=SESSION_SPLIT_MIN):
    t = hm[0] * 60 + hm[1]
    return t - 1440 if t >= split else t


# The exact close-print minute (16:00), on the session's own smin axis - see the note on ENTRY above.
# Passed as an explicit `exclude` to ENTRY's own _nearest() call so that even the FALLBACK path (16:01
# itself missing a print, search widening outward) can never land back on this known-degenerate minute -
# retargeting ENTRY to 16:01 fixes the common case, this closes the residual edge case.
POISONED_CLOSE_SMIN = _to_smin((16, 0))

# The session's own axis puts midnight (the FOLLOWING calendar day's 00:00) at smin==0 exactly (see
# rv_cell_stage.py: `late` rows get -1440, "early" ones - which 00:00 itself is - keep their raw minute,
# and 00:00 raw is 0). REBASE_TO_TODAYS_CLOSE below reads from here on to find the session's own
# just-rolled-over reference close.
_MIDNIGHT_SMIN = 0


def build_checkpoints(split=SESSION_SPLIT_MIN):
    """The seven checkpoints on one increasing minute axis, each tolerance capped at half the gap to its
    nearest neighbour so two checkpoints can never claim the same print."""
    smins = [_to_smin(hm, split) for _, hm, _ in _RAW_CHECKPOINTS]
    out = {}
    for i, (name, hm, req) in enumerate(_RAW_CHECKPOINTS):
        prev_gap = (smins[i] - smins[i - 1]) if i > 0 else math.inf
        next_gap = (smins[i + 1] - smins[i]) if i + 1 < len(smins) else math.inf
        tol = min(req, prev_gap // 2, next_gap // 2)
        out[name] = Checkpoint(name, smins[i], int(tol))
    return out


def default_cfg(**over):
    cfg = dict(
        min_dev=0.5,          # the trigger: |Stack% at 15:50| must exceed this
        gamma_floor=0.5, gamma_step=0.1, gamma_max=15.0, gamma_min_n=8, gamma_z=1.96,
        max_abs_pp=60.0,      # a print beyond this is a broken quote, not a price
    )
    cfg.update(over)
    return cfg


def panel_from_stack(tickers, t_first, stack, prev_close=None):
    """One value per ticker per minute - Stack% itself (+ optionally the raw `prev_close` it was measured
    against, same shape, for REBASE_TO_TODAYS_CLOSE below). `t_first` is the SMIN of column 0, on the same
    axis `build_checkpoints` uses."""
    out = dict(tickers=np.asarray(tickers), t_first=int(t_first), stack=np.asarray(stack, np.float32))
    if prev_close is not None:
        out["prev_close"] = np.asarray(prev_close, np.float32)
    return out


def _clean(stack, cfg):
    bad = ~np.isfinite(stack) | (np.abs(stack) > cfg["max_abs_pp"])
    out = stack.copy()
    out[bad] = np.nan
    return out


def _nearest(mat, t_first, t_col, tol, exclude=None):
    """value + the absolute minute actually used, per row, searching outward from t_col: 0, -1, +1, -2, +2,
    ... so a tie between an earlier and a later print keeps the earlier one - the first ring that finds a
    print wins, and a row already filled is never looked at again.

    `exclude`: raw minute indices (same axis as `t_col`) that are never accepted as a match even if
    finite - a known-degenerate print is skipped as if it were NaN, and the search keeps widening past it.
    See POISONED_CLOSE_SMIN.

    Sentinel: a row with NO match anywhere in the window gets `minute == -1` (the literal integer, NOT
    `-1 + t_first`) - `_read_at` below relies on this being checked as an exact value, not as "minute < 0",
    since real smin values on this axis are legitimately negative for every evening checkpoint.
    """
    N, T = mat.shape
    val = np.full(N, np.nan, np.float32)
    used = np.full(N, -1, np.int64)
    remaining = np.arange(N)
    j0 = t_col - t_first
    excl_j = {m - t_first for m in exclude} if exclude else set()
    for d in range(0, int(tol) + 1):
        if remaining.size == 0:
            break
        for j in ((j0,) if d == 0 else (j0 - d, j0 + d)):
            if remaining.size == 0:
                break
            if j < 0 or j >= T or j in excl_j:
                continue
            v = mat[remaining, j]
            hit = np.isfinite(v)
            if hit.any():
                idx = remaining[hit]
                val[idx] = v[hit]
                used[idx] = j
                remaining = remaining[~hit]
    minute = np.where(used >= 0, used + t_first, -1)
    return val, minute


def _read_at(mat, t_first, minutes):
    """Reads `mat[row, minutes[row] - t_first]` per row - the value a PREVIOUS `_nearest()` call actually
    landed on (its returned `minute`), not a fresh search. NaN where `minutes[row] == -1` (no match) or
    the column is out of the panel's own recorded range.

    BUG FIXED (self-caught, 2026-09-22): the first version gated on `minutes >= 0`, copying the sentinel
    check from the wrong side - `_nearest()`'s "no match" sentinel really is the literal `-1`, but the smin
    axis itself is legitimately negative for every EVENING checkpoint (ENTRY=-479, SIGNAL=-490,
    EXIT18=-360, EXIT21=-180), so `minutes >= 0` silently rejected every real match on that whole axis, not
    only genuine misses - `_read_at` returned all-NaN for ENTRY/EXIT18/EXIT21 even when `_nearest` DID find
    a row, which made every trade vanish the instant prev_close rebasing was switched on. Fixed by checking
    the sentinel as an exact value instead of a sign.
    """
    N, T = mat.shape
    out = np.full(N, np.nan, np.float32)
    col = minutes - t_first
    ok = (minutes != -1) & (col >= 0) & (col < T)
    idx = np.flatnonzero(ok)
    out[idx] = mat[idx, col[idx]]
    return out


def _todays_close(prev_close_mat, t_first):
    """Per ticker: the reference close in effect once the session crosses into the FOLLOWING calendar day
    (user, 2026-09-22, checked against the REAL final.parquet, not a mock: queried AMZN 2026-08-26 directly
    off `C:\\datum-api-examples-main\\OriON\\CRACEN\\final.parquet` - `prev_close` sat at 260.97 through the
    ENTIRE calendar day, including 19:40 through 23:58 (hours after the actual ~16:00 session close), and
    only jumped to 260.28 at the literal midnight rollover into 2026-08-27, a -0.26pp step - final.parquet's
    Stack% is measured against a reference that only advances at the CALENDAR date boundary, roughly 8 hours
    after the actual session close).

    "після 16:00 все повинно перераховуватися по нових відсотках" (user, 2026-09-22): every checkpoint AFTER
    the close - ENTRY and all five EXIT classes - is rebased onto THIS value (the day's own just-established
    close), not left on whatever stale reference happened to be in effect when each one's own row was
    written. SIGNAL (15:50, genuinely before the close) is NOT rebased - yesterday's close is the correct
    reference for a pre-close reading.

    Found as the first non-NaN `prev_close` at or after the session's own midnight column (smin==0, see
    _MIDNIGHT_SMIN) - the value is constant for the whole post-midnight portion of the session, so the first
    hit is the right one. NaN for a ticker with no row at all past midnight in this session (nothing to
    rebase onto - callers must drop that ticker's post-close readings rather than leave them unrebased).
    """
    N, T = prev_close_mat.shape
    out = np.full(N, np.nan, np.float32)
    start_col = max(0, _MIDNIGHT_SMIN - t_first)
    for j in range(start_col, T):
        need = np.isnan(out)
        if not need.any():
            break
        v = prev_close_mat[:, j]
        hit = need & np.isfinite(v)
        out[hit] = v[hit]
    return out


def _rebase(v, own_prev_close, target_prev_close):
    """What `v` (a Stack% reading measured against `own_prev_close`) would read if measured against
    `target_prev_close` instead - same underlying price, different denominator. NaN propagates through
    (a ticker with no known target simply cannot be rebased)."""
    return ((1.0 + v / 100.0) * (own_prev_close / target_prev_close) - 1.0) * 100.0


def scan_day(panel, cfg, checkpoints):
    """-> (trades, signals, entries) for one session.
    trades: one row per ticker per exit class that had BOTH a triggering signal (|dev|>min_dev) and a valid
    entry AND a valid exit for that class - a ticker missing one class's exit still gets the others.
    signals: the RAW 15:50 reading for every ticker that had one, gated on nothing - this is ALPHA's source
    ("the root-mean-square of ALL the deviations it gives at 15:50", not only the ones that triggered).
    entries: the RAW ENTRY-checkpoint Stack% reading (16:00/16:01) for every ticker that had one, gated on
    nothing and captured BEFORE REBASE_TO_TODAYS_CLOSE below - i.e. the value exactly as final.parquet
    carries it, still measured against the STALE (pre-rollover) reference close, "до моменту перерахунку
    клоуза". This is LAMBDA's own source (see rv_cell_ratings.py's compute_lambda): the entry-checkpoint
    counterpart to ALPHA's gate-free 15:50 population."""
    stack = _clean(panel["stack"], cfg)
    prev_close = panel.get("prev_close")
    t_first = panel["t_first"]
    tk = panel["tickers"]

    sig_v, sig_m = _nearest(stack, t_first, checkpoints["SIGNAL"].smin, checkpoints["SIGNAL"].tol)
    ent_v, ent_m = _nearest(stack, t_first, checkpoints["ENTRY"].smin, checkpoints["ENTRY"].tol,
                             exclude={POISONED_CLOSE_SMIN})

    # LAMBDA's population: the ENTRY print as-is, before the rebase a few lines down moves it onto today's
    # newly-rolled-over close. Captured here, not after, so lambda and the traded entry_val are deliberately
    # different numbers measured against different reference closes.
    ent_raw_ok = np.isfinite(ent_v)
    entries = pd.DataFrame({"ticker": tk[ent_raw_ok], "entry_raw": ent_v[ent_raw_ok]})

    today_close = None
    if prev_close is not None:
        today_close = _todays_close(prev_close, t_first)
        ent_pc = _read_at(prev_close, t_first, ent_m)
        ent_v = _rebase(ent_v, ent_pc, today_close)  # ENTRY is AFTER the close - rebase it too

    sig_ok = np.isfinite(sig_v)
    signals = pd.DataFrame({"ticker": tk[sig_ok], "dev": sig_v[sig_ok]})

    have = sig_ok & np.isfinite(ent_v) & (np.abs(sig_v) > cfg["min_dev"])
    if not have.any():
        return _EMPTY_TRADES.copy(), signals, entries

    idx0 = np.flatnonzero(have)
    entry_dev = sig_v[idx0]
    sign_key = np.where(entry_dev > 0, "pos", "neg")

    rows = []
    for cls in EXIT_CLASSES:
        ck = checkpoints[cls]
        ex_v, ex_m = _nearest(stack, t_first, ck.smin, ck.tol)
        if prev_close is not None:
            ex_pc = _read_at(prev_close, t_first, ex_m)
            ex_v = _rebase(ex_v, ex_pc, today_close)
        ex_ok = np.isfinite(ex_v[idx0])
        if not ex_ok.any():
            continue
        j = idx0[ex_ok]
        e_dev = sig_v[j]
        e_val = ent_v[j]
        x_val = ex_v[j]
        # short the high one (pos), long the low one (neg): pnl = entry - exit for a short, exit - entry for
        # a long, i.e. -sign(entry_dev) * (exit - entry) either way
        pnl = -np.sign(e_dev) * (x_val - e_val)
        rows.append(pd.DataFrame({
            "ticker": tk[j], "cls": cls, "sign": np.where(e_dev > 0, "pos", "neg"),
            "entry_dev": e_dev, "entry_val": e_val, "exit_val": x_val, "pnl": pnl.astype(np.float32),
            "signal_minute": sig_m[j], "entry_minute": ent_m[j], "exit_minute": ex_v[j] * 0 + ex_m[j],
            "date_offset": EXIT_DATE_OFFSET[cls],
        }))
    trades = pd.concat(rows, ignore_index=True) if rows else _EMPTY_TRADES.copy()
    return trades, signals, entries


_EMPTY_TRADES = pd.DataFrame(columns=["ticker", "cls", "sign", "entry_dev", "entry_val", "exit_val", "pnl",
                                       "signal_minute", "entry_minute", "exit_minute", "date_offset"])


def _add_days(sdate_int, offset_days):
    """int YYYYMMDD + a signed day offset -> int YYYYMMDD, real calendar arithmetic (not a bare -1 on the
    int, which would break across month/year boundaries)."""
    if offset_days == 0:
        return sdate_int
    ts = pd.Timestamp(str(sdate_int)) + pd.Timedelta(days=int(offset_days))
    return int(ts.strftime("%Y%m%d"))


def scan_days(day_iter, cfg, checkpoints=None):
    checkpoints = checkpoints or build_checkpoints()
    tr_all, sg_all, en_all = [], [], []
    for sdate, panel in day_iter:
        tr, sg, en = scan_day(panel, cfg, checkpoints)
        if len(tr):
            tr = tr.copy()
            # `sdate` (the SESSION id, the morning-side calendar date - see rv_cell_stage.py) is kept as its
            # own column: signals/alpha are computed per SESSION and need it. `trade_date` is the per-TRADE
            # column the wire actually dates by - see EXIT_DATE_OFFSET above for why the two can differ.
            tr.insert(0, "sdate", int(sdate))
            tr["trade_date"] = [_add_days(int(sdate), off) for off in tr["date_offset"]]
            tr_all.append(tr.drop(columns="date_offset"))
        if len(sg):
            sg = sg.copy()
            sg.insert(0, "sdate", int(sdate))
            sg_all.append(sg)
        if len(en):
            en = en.copy()
            en.insert(0, "sdate", int(sdate))
            en_all.append(en)
    trades = pd.concat(tr_all, ignore_index=True) if tr_all else (
        _EMPTY_TRADES.copy().assign(sdate=pd.Series(dtype=int), trade_date=pd.Series(dtype=int))
        [["sdate", *[c for c in _EMPTY_TRADES.columns if c != "date_offset"], "trade_date"]])
    signals = pd.concat(sg_all, ignore_index=True) if sg_all else pd.DataFrame(columns=["sdate", "ticker", "dev"])
    entries = pd.concat(en_all, ignore_index=True) if en_all else pd.DataFrame(columns=["sdate", "ticker", "entry_raw"])
    return trades, signals, entries


In [ ]:
from __future__ import annotations

# ── ratings, alpha, gamma from the trade/signal tables scan_days produces ──────────────────────────────────
import math

import numpy as np
import pandas as pd


def _wilson_lb(k, n, z=1.96):
    k = np.asarray(k, float)
    n = np.asarray(n, float)
    with np.errstate(all="ignore"):
        p = k / n
        den = 1 + z * z / n
        c = p + z * z / (2 * n)
        r = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
        return np.where(n > 0, (c - r) / den, np.nan)


def _modal_value(values, step=0.1, min_n=3):
    """The value a stack gives MOST OFTEN (user, 2026-09-22: "те відсоткове значення яке найчастіше дає
    стак"), i.e. the mode of the reading, not its RMS. |values| are binned at the same 0.1pp granularity
    GAMMA already walks in (gamma_step), the modal bin wins, and the reported value is the MEAN of the raw
    readings that landed in it (not the bin's own floor/midpoint) so the number is not quantisation-biased.
    NaN below min_n readings - a "most common value" from one or two prints is not a stat, it is an echo."""
    v = np.asarray(values, float)
    v = v[np.isfinite(v)]
    if v.size < min_n:
        return float("nan")
    bins = np.round(v / step) * step
    uniq, counts = np.unique(bins, return_counts=True)
    best = uniq[np.argmax(counts)]
    in_bin = np.abs(bins - best) < 1e-9
    return float(np.mean(v[in_bin]))


def compute_alpha(signals, step=0.1, min_n=3):
    """ALPHA, per ticker, split pos/neg (user, 2026-09-22: "воно окреме для додатнії і від'ємних відхилень"):
    the modal |15:50 reading| on each side, over EVERY reading the ticker produced - whether or not it was
    big enough to trade, same gate-free universe the old RMS definition used. -> DataFrame indexed by ticker,
    columns alpha_pos/alpha_neg."""
    if not len(signals):
        return pd.DataFrame(columns=["alpha_pos", "alpha_neg"]).rename_axis("ticker")
    out = {}
    for tk, g in signals.groupby("ticker"):
        dev = g["dev"].to_numpy()
        out[tk] = (
            _modal_value(dev[dev > 0], step, min_n),
            _modal_value(-dev[dev < 0], step, min_n),  # stored as a magnitude, like alpha_pos
        )
    return pd.DataFrame.from_dict(out, orient="index", columns=["alpha_pos", "alpha_neg"]).rename_axis("ticker")


def compute_lambda(entries, min_n=2):
    """LAMBDA (λ), per ticker: sample std (ddof=1) of the RAW ENTRY-checkpoint Stack% reading (16:00/16:01,
    "на момент 16:00, до моменту перерахунку клоуза" - user, 2026-09-27), measured against whatever
    reference close was still in effect at that print, BEFORE REBASE_TO_TODAYS_CLOSE moves it onto the
    day's newly-rolled-over close (see scan_day's own note on `entries`, and rv_cell_core.py's
    REBASE_TO_TODAYS_CLOSE for why entry_val - the traded price - is a different, rebased number).

    Gate-free, same universe as ALPHA: every session's entry print counts, whether or not |15:50 dev| ever
    cleared min_dev. NaN below min_n readings - a std from a single point is not defined. -> Series indexed
    by ticker, named "lambda"."""
    if not len(entries):
        return pd.Series(dtype=float, name="lambda").rename_axis("ticker")
    counts = entries.groupby("ticker")["entry_raw"].transform("count")
    out = entries[counts >= min_n].groupby("ticker")["entry_raw"].std(ddof=1)
    return out.rename("lambda").rename_axis("ticker")


def compute_ratings(trades):
    """total/wins/win_rate(+Wilson LB)/mean pnl(+95% LB)/median, per (ticker, cls, sign)."""
    if not len(trades):
        return pd.DataFrame(columns=["ticker", "cls", "sign", "total", "wins", "win_rate", "win_rate_lb",
                                     "mean_pnl", "median_pnl", "pnl_lb"])
    g = trades.groupby(["ticker", "cls", "sign"])
    out = g["pnl"].agg(total="size", mean_pnl="mean", median_pnl="median", sd_pnl="std").reset_index()
    out["wins"] = g["pnl"].apply(lambda s: int((s > 0).sum())).to_numpy()
    out["win_rate"] = out["wins"] / out["total"]
    out["win_rate_lb"] = _wilson_lb(out["wins"], out["total"])
    with np.errstate(all="ignore"):
        se = out["sd_pnl"] / np.sqrt(out["total"])
        out["pnl_lb"] = np.where(out["total"] > 1, out["mean_pnl"] - 1.96 * se, np.nan)
    return out.drop(columns="sd_pnl")


def compute_gamma(trades, cfg):
    """GAMMA, per (ticker, cls, sign): the LOWEST level L >= gamma_floor at which trades reaching |entry_dev|
    >= L have a positive 95% lower bound on mean pnl, walking up from the floor in gamma_step increments and
    stopping at the first level that qualifies OR the first level with too little evidence (< gamma_min_n).
    Lowest, because frequency only falls as L rises - "the value the ticker gives MOST OFTEN that is > 0.5
    and successful" is exactly the first (most populated) level that clears the bar. None when nothing does."""
    if not len(trades):
        return pd.DataFrame(columns=["ticker", "cls", "sign", "gamma", "gamma_n", "gamma_mean", "gamma_rate"])
    rows = []
    for (tk, cls, sg), g in trades.groupby(["ticker", "cls", "sign"]):
        dev = np.abs(g["entry_dev"].to_numpy())
        pnl = g["pnl"].to_numpy()
        lvl = float(cfg["gamma_floor"])
        found = None
        while lvl <= cfg["gamma_max"] + 1e-9:
            m = dev >= lvl - 1e-9
            n = int(m.sum())
            if n < cfg["gamma_min_n"]:
                break
            mp = float(pnl[m].mean())
            se = float(pnl[m].std(ddof=1)) / math.sqrt(n) if n > 1 else math.inf
            lb = mp - cfg["gamma_z"] * se
            if lb > 0:
                found = (lvl, n, mp, float((pnl[m] > 0).mean()))
                break
            lvl = round(lvl + cfg["gamma_step"], 6)
        if found is not None:
            rows.append({"ticker": tk, "cls": cls, "sign": sg, "gamma": found[0], "gamma_n": found[1],
                        "gamma_mean": found[2], "gamma_rate": found[3]})
    return pd.DataFrame(rows, columns=["ticker", "cls", "sign", "gamma", "gamma_n", "gamma_mean", "gamma_rate"])


def build_summary(trades, signals, cfg, sigma_by_ticker=None, lambda_by_ticker=None):
    """alpha + ratings + gamma, joined into one per-(ticker, cls, sign) table.

    Each row's "alpha" is the SIGN-MATCHING side of the split alpha (a "pos" row - a short, born of a high
    15:50 reading - carries alpha_pos; a "neg" row carries alpha_neg), not a single ticker-wide scalar
    anymore. `sigma` is a per-ticker STATIC value already published elsewhere (user, 2026-09-22: "Сігма це
    статичне значення кожного стака воно є в файлі") - passed in from the stage step's own column read
    (rv_cell_stage.py), not computed here; a ticker with no static sigma seen gets NaN, same as a ticker
    with no gamma. `lambda` is `compute_lambda`'s own per-ticker Series (pre-rebase ENTRY-checkpoint std),
    passed in the same shape as sigma_by_ticker but computed from this run's own entries table, not read
    from a file.
    """
    alpha = compute_alpha(signals)
    ratings = compute_ratings(trades)
    gamma = compute_gamma(trades, cfg)
    out = ratings.merge(gamma, on=["ticker", "cls", "sign"], how="left")

    alpha_pos = out["ticker"].map(alpha["alpha_pos"]) if len(alpha) else np.nan
    alpha_neg = out["ticker"].map(alpha["alpha_neg"]) if len(alpha) else np.nan
    out["alpha"] = np.where(out["sign"] == "pos", alpha_pos, alpha_neg)

    if sigma_by_ticker:
        out["sigma"] = out["ticker"].map(sigma_by_ticker)
    else:
        out["sigma"] = np.nan

    if lambda_by_ticker is not None and len(lambda_by_ticker):
        out["lambda"] = out["ticker"].map(lambda_by_ticker)
    else:
        out["lambda"] = np.nan

    return out, alpha


In [ ]:
from __future__ import annotations

# ── stage: final.parquet -> one small parquet per SESSION (15:00 one day -> 15:00 the next) ────────────────
#
# The day is split at 15:00 - strictly before the 15:50 signal and well after the 09:30 PRINT checkpoint of
# the PREVIOUS session - so every one of 15:50, 16:00, 18:00, 21:00 (this calendar evening) and 04:00, 07:00,
# 09:30 (the FOLLOWING calendar morning) lands in the SAME session file. This is what "переходить через
# опівніч" means here: the stage does the midnight crossing once, so scan_day never has to.
import numpy as np
import pandas as pd

# SESSION_SPLIT_MIN and panel_from_stack come from rv_cell_core.py, executed into the same namespace before
# this cell (the notebook order; the test harness below mirrors it) - not imported, same pattern as
# Pullback.ipynb's stage cell reusing panel_from_stack from its own core cell.


def rv_stage_by_session(input_path, stage_dir, *, session_split_min=SESSION_SPLIT_MIN, start_date=None,
                        STOCK_NUM_FIELD="Stack%", SIGMA_FIELD="sigma", PREV_CLOSE_FIELD="prev_close",
                        smin_lo=-520, smin_hi=650, log_every_n_chunks=20):
    """-> (sorted session dates, {ticker: static sigma}).

    Sigma is READ, not computed: the source final.parquet already carries a per-row "sigma" column, the
    same static-per-ticker dispersion figure ArbitRage.ipynb's own stage reads off "sigma"/"corr"/"beta"
    columns of the identical file (user, 2026-09-22: "Сігма це статичне значення кожного стака воно є в
    файлі"). The LATEST non-null value seen per ticker wins (row groups are read in file order, which is
    time order), so a ticker's sigma tracks the most recently known dispersion rather than freezing on
    whatever the very first row happened to hold.

    `prev_close` is ALSO staged now, per-minute alongside `stack`, so `scan_day`'s REBASE_EXIT_TO_ENTRY_
    BASIS can read the exact reference close each checkpoint's Stack% was measured against and correct for
    final.parquet's own close-reference-only-rolls-at-midnight artifact (see rv_cell_core.py's own note on
    this - verified 2026-09-22 against the real file). It is written as its own column in the staged
    parquet (not folded into `stack`) so a reader that does not need it is not forced to carry it.
    """
    import gc, time, shutil
    import pyarrow as pa
    import pyarrow.parquet as pq
    from pathlib import Path

    stage = Path(stage_dir)
    if stage.exists():
        shutil.rmtree(stage)
    stage.mkdir(parents=True, exist_ok=True)
    start_i = int(start_date.replace("-", "")) if start_date else -1

    schema = pa.schema([("ticker", pa.string()), ("smin", pa.int16()), ("stack", pa.float32()),
                        ("prev_close", pa.float32())])
    writers, counts = {}, {}
    sigma_by_ticker: dict = {}

    def _writer(sdate):
        if sdate not in writers:
            writers[sdate] = pq.ParquetWriter(str(stage / f"{sdate}.parquet"), schema, compression="zstd")
            counts[sdate] = 0
        return writers[sdate]

    pf = pq.ParquetFile(input_path)
    want = ["ticker", "dt", STOCK_NUM_FIELD]
    cols = [c for c in want if c in pf.schema.names]
    for need in want:
        if need not in cols:
            raise KeyError(f"final.parquet is missing required column: {need}")
    has_sigma = SIGMA_FIELD in pf.schema.names
    if has_sigma:
        cols = cols + [SIGMA_FIELD]
    else:
        print(f"WARN: final.parquet has no '{SIGMA_FIELD}' column - sigma will be null for every ticker")
    has_prev_close = PREV_CLOSE_FIELD in pf.schema.names
    if has_prev_close:
        cols = cols + [PREV_CLOSE_FIELD]
    else:
        print(f"WARN: final.parquet has no '{PREV_CLOSE_FIELD}' column - REBASE_EXIT_TO_ENTRY_BASIS is "
              f"disabled, EXIT04/EXIT07/PRINT pnl will carry the midnight-rollover contamination unfixed")
    t0 = time.time()
    total_in = total_out = 0
    print(f"START reversal stage  file={input_path}  session_split={session_split_min}min")
    try:
        for ci in range(pf.num_row_groups):
            df = pf.read_row_group(ci, columns=cols).to_pandas()
            total_in += len(df)
            dt = pd.to_datetime(df["dt"], errors="coerce", utc=True)
            ok = dt.notna().to_numpy(copy=False)
            if not ok.any():
                continue
            dt, df = dt[ok], df.loc[ok]
            t_arr = dt.dt.hour.to_numpy(dtype="int32", copy=False) * 60 + dt.dt.minute.to_numpy(dtype="int32", copy=False)
            late = t_arr >= session_split_min
            smin = np.where(late, t_arr - 1440, t_arr).astype("int16")
            sess = dt + pd.to_timedelta(np.where(late, 1, 0), unit="D")
            sdate = (sess.dt.year.to_numpy(dtype="int32", copy=False) * 10000 + sess.dt.month.to_numpy(dtype="int32", copy=False) * 100
                     + sess.dt.day.to_numpy(dtype="int32", copy=False)).astype("int32")
            stack = pd.to_numeric(df[STOCK_NUM_FIELD], errors="coerce").to_numpy(dtype="float32", copy=False)
            if has_prev_close:
                prev_close = pd.to_numeric(df[PREV_CLOSE_FIELD], errors="coerce").to_numpy(dtype="float32", copy=False)
            else:
                prev_close = np.full(len(df), np.nan, dtype="float32")
            if has_sigma:
                sigma_col = pd.to_numeric(df[SIGMA_FIELD], errors="coerce").to_numpy(dtype="float64", copy=False)
                tick_col = df["ticker"].to_numpy(copy=False)
                for tk, sv in zip(tick_col, sigma_col):
                    if np.isfinite(sv):
                        sigma_by_ticker[str(tk)] = float(sv)
            keep = (smin >= smin_lo) & (smin <= smin_hi) & np.isfinite(stack)
            if start_i > 0:
                keep &= sdate >= start_i
            if not keep.any():
                continue
            out = pd.DataFrame({"ticker": df["ticker"].to_numpy(copy=False)[keep].astype(str), "sdate": sdate[keep],
                                "smin": smin[keep], "stack": stack[keep], "prev_close": prev_close[keep]})
            for sd, part in out.groupby("sdate", sort=False):
                _writer(int(sd)).write_table(pa.Table.from_pandas(part[["ticker", "smin", "stack", "prev_close"]], schema=schema, preserve_index=False))
                counts[int(sd)] += len(part)
                total_out += len(part)
            del df, out
            if (ci + 1) % log_every_n_chunks == 0:
                print(f"[rg {ci + 1:>4}/{pf.num_row_groups}] in={total_in:,} staged={total_out:,} sessions={len(writers)} elapsed={time.time() - t0:.1f}s")
                gc.collect()
    finally:
        for w in writers.values():
            w.close()
    print(f"DONE stage in={total_in:,} staged={total_out:,} sessions={len(writers)} "
          f"sigma_tickers={len(sigma_by_ticker):,} elapsed={time.time() - t0:.1f}s")
    return sorted(counts), sigma_by_ticker


def rv_sessions(stage_dir, *, smin_lo=-520, smin_hi=650, min_tickers=30):
    """yields (sdate, panel) in date order, one pivoted session at a time."""
    from pathlib import Path
    for f in sorted(Path(stage_dir).glob("*.parquet"), key=lambda p: int(p.stem)):
        df = pd.read_parquet(f)
        if df["ticker"].nunique() < min_tickers:
            continue
        df = df.drop_duplicates(["ticker", "smin"], keep="last")
        tick, ti = np.unique(df["ticker"].to_numpy(), return_inverse=True)
        T = smin_hi - smin_lo + 1
        stack = np.full((tick.size, T), np.nan, np.float32)
        stack[ti, df["smin"].to_numpy().astype(np.int64) - smin_lo] = df["stack"].to_numpy()
        prev_close = None
        if "prev_close" in df.columns:
            prev_close = np.full((tick.size, T), np.nan, np.float32)
            prev_close[ti, df["smin"].to_numpy().astype(np.int64) - smin_lo] = df["prev_close"].to_numpy()
        yield int(f.stem), panel_from_stack(tick, smin_lo, stack, prev_close)


In [ ]:
from pathlib import Path
import os, gzip, json, datetime


def _resolve_orion_paths(strategy_code: str):
    final_env = os.environ.get("FINAL_PARQUET_PATH")
    sig_env = os.environ.get("SIGNALS_DIR")
    orion_env = os.environ.get("ORION_HOME")

    final_path = Path(final_env).expanduser().resolve() if final_env else None
    signals_base = Path(sig_env).expanduser().resolve() if sig_env else None

    if (final_path is None or signals_base is None) and orion_env:
        orion_home = Path(orion_env).expanduser().resolve()
        if final_path is None:
            final_path = (orion_home / "CRACEN" / "final.parquet").resolve()
        if signals_base is None:
            signals_base = (orion_home / "signals").resolve()

    if final_path is None or signals_base is None:
        here = Path.cwd().resolve()
        orion_home = None
        for parent in [here] + list(here.parents):
            if parent.name.lower() == "orion":
                orion_home = parent
                break
            cand = parent / "OriON"
            if cand.exists() and cand.is_dir():
                orion_home = cand.resolve()
                break
        if orion_home is None:
            raise RuntimeError("Cannot locate OriON. Set ORION_HOME env var (recommended).")
        if final_path is None:
            final_path = (orion_home / "CRACEN" / "final.parquet").resolve()
        if signals_base is None:
            signals_base = (orion_home / "signals").resolve()

    out_dir = (signals_base / strategy_code.lower()).resolve()
    out_dir.mkdir(parents=True, exist_ok=True)
    if not final_path.exists():
        raise FileNotFoundError(f"FINAL parquet not found: {final_path}")
    return final_path, out_dir


# ── runner ───────────────────────────────────────────────────────────────────────────────────────────────
FINAL_PATH, OUT_DIR = _resolve_orion_paths("reversal")

# the per-session stage lives OUTSIDE signals/ - run_orion_daily.py publishes the whole signals/ tree
STAGE_DIR = OUT_DIR.parent.parent / "_work" / "reversal_stage"
STAGE_DIR.parent.mkdir(parents=True, exist_ok=True)
SIGMA_SIDECAR = STAGE_DIR.parent / "reversal_stage_sigma.json"

_stage_ok = STAGE_DIR.exists() and any(STAGE_DIR.glob("*.parquet")) and SIGMA_SIDECAR.exists() and \
    min(p.stat().st_mtime for p in STAGE_DIR.glob("*.parquet")) >= FINAL_PATH.stat().st_mtime
# The sidecar's own existence is now PART of freshness, not just read after the fact: a stage directory
# left over from before sigma was added (every v1 run, and the very first v2 run against an old cache) is
# "newer than final.parquet" by mtime alone, so the old check reused it and silently produced sigma=null
# for EVERY ticker - measured on the 2026-09-22 real republish: alpha_pos/alpha_neg populated for 7320/7402
# tickers, sigma for 0/7402. Requiring the sidecar forces exactly ONE fresh re-stage the first time this
# runs against a pre-existing cache (capturing sigma), after which the sidecar exists and reuse is sound.
if _stage_ok:
    print("stage is at least as new as final.parquet (incl. its sigma sidecar) - reusing", STAGE_DIR)
    sigma_by_ticker = json.loads(SIGMA_SIDECAR.read_text(encoding="utf-8"))
else:
    _, sigma_by_ticker = rv_stage_by_session(str(FINAL_PATH), str(STAGE_DIR))
    SIGMA_SIDECAR.write_text(json.dumps(sigma_by_ticker), encoding="utf-8")
    if not sigma_by_ticker:
        print("WARN: fresh stage still produced no sigma for any ticker - check final.parquet's own "
              "'sigma' column (see rv_stage_by_session's own WARN above if the column is missing entirely)")

CFG = default_cfg()
CHECKPOINTS = build_checkpoints()

trades, signals, entries = scan_days(rv_sessions(str(STAGE_DIR)), CFG, CHECKPOINTS)
print(f"trades: {len(trades):,} over {trades['sdate'].nunique() if len(trades) else 0} sessions, "
      f"{trades['ticker'].nunique() if len(trades) else 0} tickers")

# LAMBDA is computed fresh every run from THIS run's own entries table (not read from a sidecar like sigma
# is) - it is cheap (one groupby over an already-in-memory table) and, unlike sigma, has no independent
# source file to go stale against.
lambda_by_ticker = compute_lambda(entries)
print(f"lambda: {len(lambda_by_ticker):,} tickers")

summary, alpha = build_summary(trades, signals, CFG, sigma_by_ticker, lambda_by_ticker)
summary.to_csv(OUT_DIR / "reversal_ratings.csv", index=False)

# ── the scout file: ONE file a Scout page reads exactly the way ArbitrageScout/PairFluxScout do -
# a per-ticker row carrying alpha, the per-(class,sign) rating+gamma block ("classes", the same shape
# best_params.jsonl.gz's per-ticker block has - PnlBasis / windows / recentDates in meta so the bridge
# never has to guess), and the trade log itself ("perevirity vidhylennya 1 alpha") ──
# Sessions ("sdate") are the SIGNAL/ENTRY axis; trades date by EXIT (see EXIT_DATE_OFFSET in rv_cell_core.py),
# so a trade's own "date" can fall a calendar day either side of its session - recent_dates/date_index are
# built from trade_date's own union, not from sdate, or an EXIT18 trade dated one day before its session
# would have nowhere to point.
trade_dates_sorted = sorted(trades["trade_date"].unique().tolist()) if len(trades) else []
recent_dates = [str(s) for s in trade_dates_sorted][-255:]  # byte-indexed on the wire, like Arbitrage's own date column
date_index = {int(d): i for i, d in enumerate(recent_dates)}
WINDOWS_DAYS = [5, 20, 40, 65]

CLASS_ORDER = ["EXIT18", "EXIT21", "EXIT04", "EXIT07", "PRINT"]


def _rate(r):
    return None if pd.isna(r) else round(float(r), 4)


def _lambda_of(tk):
    return None if tk not in lambda_by_ticker.index else round(float(lambda_by_ticker[tk]), 4)


ROLLING_PATH = OUT_DIR / "reversal_rolling_perf.json.gz"
# "ratings": a FLAT array of self-describing cells (like "trades"), not a nested {cls:{sign:...}} object -
# one shape for the streaming parser to know, the same way every other array in this file works.
by_ticker_ratings: dict = {}
for _, r in summary.iterrows():
    by_ticker_ratings.setdefault(r["ticker"], []).append({
        "cls": r["cls"], "sign": r["sign"], "total": int(r["total"]), "wins": int(r["wins"]),
        "win_rate": _rate(r["win_rate"]), "win_rate_lb": _rate(r["win_rate_lb"]),
        "mean_pnl": _rate(r["mean_pnl"]), "pnl_lb": _rate(r["pnl_lb"]),
        "gamma": _rate(r["gamma"]), "gamma_n": None if pd.isna(r["gamma_n"]) else int(r["gamma_n"]),
        "gamma_rate": _rate(r["gamma_rate"]),
    })


def _alpha_side(tk, col):
    if tk not in alpha.index:
        return None
    v = alpha.loc[tk, col]
    return None if pd.isna(v) else round(float(v), 4)


rows = []
for tk, g in trades.groupby("ticker") if len(trades) else []:
    rows.append({
        "ticker": tk,
        "alpha_pos": _alpha_side(tk, "alpha_pos"), "alpha_neg": _alpha_side(tk, "alpha_neg"),
        "sigma": None if tk not in sigma_by_ticker else round(float(sigma_by_ticker[tk]), 4),
        "lambda": _lambda_of(tk),
        "ratings": by_ticker_ratings.get(tk, []),
        "trades": [{
            "cls": r["cls"], "sign": r["sign"], "date": date_index.get(int(r["trade_date"])),
            "entry_dev": round(float(r["entry_dev"]), 4), "pnl_pct": round(float(r["pnl"]), 4),
            "status": "win" if r["pnl"] > 0 else "loss",
            "signal_minute_idx": int(r["signal_minute"]), "entry_minute_idx": int(r["entry_minute"]),
            "exit_minute_idx": int(r["exit_minute"]),
        } for _, r in g.iterrows() if int(r["trade_date"]) in date_index],
    })
# a ticker with a 15:50 reading but never a trade still carries its alpha (the scout page shows it
# with 0 trades rather than pretending it does not exist)
seen = {r["ticker"] for r in rows}
for tk in alpha.index:
    if tk in seen:
        continue
    ap, an = _alpha_side(tk, "alpha_pos"), _alpha_side(tk, "alpha_neg")
    if ap is None and an is None:
        continue
    rows.append({
        "ticker": tk, "alpha_pos": ap, "alpha_neg": an,
        "sigma": None if tk not in sigma_by_ticker else round(float(sigma_by_ticker[tk]), 4),
        "lambda": _lambda_of(tk),
        "ratings": by_ticker_ratings.get(tk, []), "trades": [],
    })

with gzip.open(ROLLING_PATH, "wt", encoding="utf-8", newline="\n") as f:
    json.dump({
        "meta": {
            "version": "reversal-rolling-perf-v2",
            "generated_at": datetime.datetime.utcnow().isoformat() + "Z",
            "most_recent_session": recent_dates[-1] if recent_dates else None,
            "sessions_seen": len(recent_dates),
            "windows_days": WINDOWS_DAYS,
            "recent_dates": recent_dates,
            "position_usd": 1000,
            "min_dev": CFG["min_dev"],
            "pnl_basis": "raw Stack% move, entry(16:00, vs the PRIOR regular-session close) -> class exit "
                        "(18:00/21:00 same evening, 04:00/07:00/09:30=PRINT next morning); short on a high "
                        "(pos) 15:50 reading, long on a low (neg) one. Trades date by their EXIT's own "
                        "calendar day, not the session's.",
            "lambda_basis": "per-ticker std(ddof=1) of the raw ENTRY-checkpoint Stack% reading (16:00/16:01), "
                        "measured BEFORE it is rebased onto the day's newly-rolled-over close - i.e. still "
                        "against the stale pre-rollover reference close final.parquet carries at that print. "
                        "Gate-free, same population as alpha.",
            "checkpoints": {k: [v.smin, v.tol] for k, v in CHECKPOINTS.items()},
        },
        "rows": rows,
    }, f, ensure_ascii=False)

print("written ->", OUT_DIR, "(", "reversal_ratings.csv", ROLLING_PATH.name, ")")
